<a href="https://colab.research.google.com/github/Abhishekbelwal/Machine-Learning-basic-algorithms-and-projects/blob/main/Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, recall_score, precision_score, f1_score
from sklearn.linear_model import LogisticRegression

In [3]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression # Added for Linear Regression
from sklearn.metrics import mean_squared_error, r2_score # Added for regression evaluation

In [4]:
california_housing = fetch_california_housing(as_frame=True)
X = california_housing.data
y = california_housing.target

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
display(X.head())

Features (X) shape: (20640, 8)
Target (y) shape: (20640,)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


### Transform Target for Logistic Regression
Since `LogisticRegression` is a classification algorithm, we need to convert the continuous `MedHouseVal` target into a binary classification problem. We'll define 'expensive' houses as those with a median value above or equal to the overall median, and 'not expensive' otherwise.

In [10]:
median_house_value = y.median()
y_binary = (y >= median_house_value).astype(int)

print(f"Median House Value: {median_house_value:.2f}")
print("Original target (y) head:")
display(y.head())
print("Binary target (y_binary) head:")
display(y_binary.head())
print(f"Distribution of binary target:\n{y_binary.value_counts()}")

Median House Value: 1.80
Original target (y) head:


,MedHouseVal
0,4.526
1,3.585
2,3.521
3,3.413
4,3.422


Binary target (y_binary) head:


,MedHouseVal
0,1
1,1
2,1
3,1
4,1


Distribution of binary target:
MedHouseVal
1    10325
0    10315
Name: count, dtype: int64


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (16512, 8)
X_test shape: (4128, 8)
y_train shape: (16512,)
y_test shape: (4128,)


### Split Data with New Binary Target

In [12]:
X_train, X_test, y_train_binary, y_test_binary = train_test_split(X, y_binary, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train_binary shape:", y_train_binary.shape)
print("y_test_binary shape:", y_test_binary.shape)

X_train shape: (16512, 8)
X_test shape: (4128, 8)
y_train_binary shape: (16512,)
y_test_binary shape: (4128,)


In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier inspection (optional)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Scaled X_train head:")
display(X_train_scaled.head())

Scaled X_train head:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,-0.326196,0.348490,-0.174916,-0.208365,0.768276,0.051376,-1.372811,1.272587
1,-0.035843,1.618118,-0.402835,-0.128530,-0.098901,-0.117362,-0.876696,0.709162
2,0.144701,-1.952710,0.088216,-0.257538,-0.449818,-0.032280,-0.460146,-0.447603
3,-1.017864,0.586545,-0.600015,-0.145156,-0.007434,0.077507,-1.382172,1.232698
4,-0.171488,1.142008,0.349007,0.086624,-0.485877,-0.068832,0.532084,-0.108551


### Feature Scaling (using previously scaled X_train_scaled and X_test_scaled)

In [14]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("Model coefficients:", model.coef_)
print("Model intercept:", model.intercept_)

Model coefficients: [ 0.85438303  0.12254624 -0.29441013  0.33925949 -0.00230772 -0.0408291
 -0.89692888 -0.86984178]
Model intercept: 2.0719469373788777


### Train a Logistic Regression Model

In [15]:
logistic_model = LogisticRegression(random_state=42, solver='liblinear') # Using liblinear for small datasets and L1/L2 regularization
logistic_model.fit(X_train_scaled, y_train_binary)

print("Logistic Regression Model coefficients:", logistic_model.coef_)
print("Logistic Regression Model intercept:", logistic_model.intercept_)

Logistic Regression Model coefficients: [[ 2.58268843  0.26725927 -0.8565319   0.95900434  0.05856863 -2.58687888
  -3.59973945 -3.39859485]]
Logistic Regression Model intercept: [0.04750816]


### Make Predictions and Evaluate the Logistic Regression Model

In [16]:
from sklearn.metrics import roc_auc_score

y_pred_binary = logistic_model.predict(X_test_scaled)
y_pred_proba = logistic_model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test_binary, y_pred_binary)
recall = recall_score(y_test_binary, y_pred_binary)
precision = precision_score(y_test_binary, y_pred_binary)
f1 = f1_score(y_test_binary, y_pred_binary)
roc_auc = roc_auc_score(y_test_binary, y_pred_proba)
conf_matrix = confusion_matrix(y_test_binary, y_pred_binary)

print(f"Accuracy Score: {accuracy:.4f}")
print(f"Recall Score: {recall:.4f}")
print(f"Precision Score: {precision:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}")
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", classification_report(y_test_binary, y_pred_binary))

Accuracy Score: 0.8249
Recall Score: 0.8241
Precision Score: 0.8237
F1 Score: 0.8239
ROC AUC Score: 0.9078
Confusion Matrix:
 [[1714  362]
 [ 361 1691]]
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.83      0.83      2076
           1       0.82      0.82      0.82      2052

    accuracy                           0.82      4128
   macro avg       0.82      0.82      0.82      4128
weighted avg       0.82      0.82      0.82      4128



In [19]:
from sklearn.metrics import classification_report

print("Classification Report:\n", classification_report(y_test_binary, y_pred_binary))

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.83      0.83      2076
           1       0.82      0.82      0.82      2052

    accuracy                           0.82      4128
   macro avg       0.82      0.82      0.82      4128
weighted avg       0.82      0.82      0.82      4128



In [17]:
y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared (R2) Score: {r2:.4f}")

Mean Squared Error (MSE): 0.5559
R-squared (R2) Score: 0.5758
